In [0]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql import functions as F

sc = SparkSession.builder\
        .appName("PySpark Eli!!")\
        .getOrCreate()

In [0]:
rdd = sc.read.text('/Volumes/workspace/default/volume/sample.txt').withColumn("row_id", F.monotonically_increasing_id())

In [0]:
rdd.collect()

In [0]:
rdd_split = rdd.withColumn('number', F.split(F.trim(F.col('value')), r'\s+'))\
               .withColumn('numbers', F.expr('TRANSFORM(number, n -> CAST(n as Integer))'))

exploded = rdd_split.withColumn('num', F.explode(F.col('numbers')))\
                    .select('row_id', 'num')

row_sums = exploded.groupBy('row_id').agg(F.sum('num').alias('row_sum'))

result = rdd_split.join(row_sums, on='row_id', how='left')\
                  .select('row_id', 'value', 'row_sum')

result.show()

In [0]:
rdd.schema

In [0]:


df_with_test = rdd.withColumn(
    "test",
    F.expr("AGGREGATE(SPLIT(TRIM(value), ' '), 0, (acc, x) -> acc + CAST(x AS INT))")
)

df_with_test.show()

## Quiz

In [0]:
quiz_text = sc.read.text('/Volumes/workspace/default/volume/quiz.txt')

result_df = quiz_text.withColumn('quiz', F.expr("TRANSFORM(SPLIT(value, ' '), w -> LEN(w))")).collect()



In [0]:
[row['quiz'] for row in result_df]